# Notebook 03v2: Train Autoencoder + B Matrix (Bug-Fixed)

Changes from v1:
- Uses v2 hidden states (500 samples, enable_thinking=False)
- Group Lasso regularization (default)
- Validates alpha distribution quality

In [ ]:
import os, sys, subprocess
REPO = 'thoughtcomm'
if os.path.exists(REPO):
    os.chdir(REPO)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AUMEZAK/thoughtcomm.git'], check=True)
    os.chdir(REPO)
    subprocess.run(['pip', 'install', '-e', '.', '-q'], check=True)

In [ ]:
import torch
import os
from configs.config import ThoughtCommConfig
from models.autoencoder import SparsityRegularizedAE
from training.train_autoencoder import train_autoencoder
from training.jacobian_utils import compute_full_jacobian, extract_binary_jacobian

from google.colab import drive
drive.mount('/content/drive')

config = ThoughtCommConfig.for_qwen_0_6b()
MODEL_TAG = config.model_name.replace('/', '_')
SAVE_DIR = config.save_dir

print(f'AE config: n_h={config.n_h}, n_z={config.n_z}, hidden={config.ae_hidden}')
print(f'Regularization: {config.jacobian_reg_type}, lambda={config.jacobian_l1_weight}')

In [ ]:
# Load v2 hidden states
math_path = os.path.join(SAVE_DIR, f'{MODEL_TAG}_math_hidden_v2.pt')
gsm8k_path = os.path.join(SAVE_DIR, f'{MODEL_TAG}_gsm8k_hidden_v2.pt')

math_data = torch.load(math_path, weights_only=False)
gsm8k_data = torch.load(gsm8k_path, weights_only=False)

H_math = math_data['H']
H_gsm8k = gsm8k_data['H']

# Combine for AE training
H_train = torch.cat([H_math, H_gsm8k], dim=0)
print(f'Combined training data: {H_train.shape}')
print(f'Mean: {H_train.float().mean():.4f}, Std: {H_train.float().std():.4f}')

In [ ]:
# Train autoencoder
ae = SparsityRegularizedAE(config.n_h, config.n_z, config.ae_hidden, config.ae_num_layers)
ae, loss_history, norm_stats = train_autoencoder(ae, H_train, config, verbose=True)

print(f'\nFinal losses:')
print(f'  rec_loss: {loss_history[-1][0]:.6f}')
print(f'  jac_loss: {loss_history[-1][1]:.6f}')
print(f'  total:    {loss_history[-1][2]:.6f}')

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
rec = [l[0] for l in loss_history]
jac = [l[1] for l in loss_history]
total = [l[2] for l in loss_history]

axes[0].plot(rec); axes[0].set_title('Reconstruction Loss'); axes[0].set_yscale('log')
axes[1].plot(jac); axes[1].set_title('Jacobian Loss')
axes[2].plot(total); axes[2].set_title('Total Loss'); axes[2].set_yscale('log')
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig(f'results/03v2_ae_training_{MODEL_TAG}.png', dpi=100)
plt.show()

In [ ]:
# Compute full Jacobian and B matrix
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ae_device = ae.to(device)

# Sample a batch for Jacobian computation
sample = H_train[:64].float().to(device)
Z_sample = ae_device.encode(sample)

print('Computing full Jacobian...')
J = compute_full_jacobian(ae_device.decoder, Z_sample)
B = extract_binary_jacobian(J, threshold=config.jacobian_threshold)

print(f'B matrix shape: {B.shape}')
print(f'Sparsity: {(B == 0).float().mean():.4f} ({(B == 0).sum()}/{B.numel()})')
print(f'Non-zero: {(B != 0).sum()}/{B.numel()}')

In [ ]:
# Analyze agreement structure
from pipeline.agreement import AgreementReweighter

reweighter = AgreementReweighter(B, config)
stats = reweighter.get_agreement_stats()

print('Agreement distribution:')
for k, v in stats['agreement_distribution'].items():
    pct = v / stats['total_dims'] * 100
    print(f'  {k}: {v} ({pct:.1f}%)')

print(f'\nPer-agent relevant dims: {stats["per_agent_relevant"]}')

# Quality check
alpha3_pct = stats['agreement_distribution'].get('alpha=3', 0) / stats['total_dims'] * 100
if alpha3_pct > 80:
    print(f'\n*** WARNING: {alpha3_pct:.0f}% alpha=3 — reweighting will have minimal effect ***')
elif alpha3_pct > 50:
    print(f'\n** NOTE: {alpha3_pct:.0f}% alpha=3 — moderate differentiation **')
else:
    print(f'\nGood: {alpha3_pct:.0f}% alpha=3 — strong differentiation')

In [ ]:
# Visualize B matrix
plt.figure(figsize=(10, 6))
B_vis = B.float().cpu()
# Show first 100 columns for visibility
plt.imshow(B_vis[:, :100].numpy(), aspect='auto', cmap='binary')
plt.xlabel('Latent dimension (first 100)')
plt.ylabel('Hidden state dimension')
plt.title(f'B(J_f) — Sparsity={((B==0).float().mean()*100):.1f}%')
# Mark agent boundaries
for i in range(1, config.num_agents):
    plt.axhline(y=i * config.hidden_size, color='red', linewidth=1, linestyle='--')
plt.colorbar()
plt.savefig(f'results/03v2_b_matrix_{MODEL_TAG}.png', dpi=100)
plt.show()

In [ ]:
# Save AE, B matrix, reweighter
ae_save_dir = os.path.join(SAVE_DIR, f'{MODEL_TAG}_ae_v2')
os.makedirs(ae_save_dir, exist_ok=True)

torch.save(ae.state_dict(), os.path.join(ae_save_dir, 'ae.pt'))
torch.save(B, os.path.join(ae_save_dir, 'B.pt'))
torch.save(reweighter.state_dict(), os.path.join(ae_save_dir, 'reweighter.pt'))

import json
summary = {
    'model': config.model_name,
    'ae_final_rec_loss': loss_history[-1][0],
    'ae_final_jac_loss': loss_history[-1][1],
    'ae_final_total_loss': loss_history[-1][2],
    'b_shape': list(B.shape),
    'b_sparsity': (B == 0).float().mean().item(),
    'b_nonzero': int((B != 0).sum()),
    'b_total': int(B.numel()),
    'agreement_stats': stats,
}
with open(f'results/03v2_autoencoder_{MODEL_TAG}.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved: ae.pt, B.pt, reweighter.pt')
print(f'Summary: results/03v2_autoencoder_{MODEL_TAG}.json')